# Обучение YOLOv8n в Yandex DataSphere

Параметры: `epochs=100`, `batch=16`, `imgsz=640`, device = GPU (если доступен).

Перед запуском:
1. Выберите конфигурацию **g1.1** (GPU) при старте вычислений.
2. Загрузите в корень проекта `processed.zip` и файлы репозитория (`ml/train.py`, `backend/`).
3. Выполняйте ячейки сверху вниз.

## 1. Установка зависимостей

In [ ]:
%pip install -q ultralytics

## 2. Проверка окружения и GPU

In [ ]:
import ultralytics

ultralytics.checks()

import torch

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")

## 3. Распаковка датасета

Ожидается архив `processed.zip` с папками `images/` и `labels/`.
После распаковки создаётся portable `data.yaml` (без абсолютных Windows-путей).

In [ ]:
from pathlib import Path
import shutil
import zipfile

# Корень проекта: каталог с ml/ или текущая рабочая директория ноутбука
cwd = Path.cwd().resolve()
if (cwd / "ml" / "train.py").is_file():
    PROJECT_ROOT = cwd
elif (cwd.parent / "ml" / "train.py").is_file():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data" / "processed"
ZIP_CANDIDATES = [
    PROJECT_ROOT / "processed.zip",
    PROJECT_ROOT / "data" / "processed.zip",
    cwd / "processed.zip",
]

zip_path = next((p for p in ZIP_CANDIDATES if p.is_file()), None)

if zip_path is not None:
    print(f"Распаковываю {zip_path} -> {DATA_DIR}")
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    # На случай архива с вложенным processed/
    nested = DATA_DIR / "processed"
    if (nested / "images").is_dir() and not (DATA_DIR / "images").is_dir():
        for child in nested.iterdir():
            shutil.move(str(child), str(DATA_DIR / child.name))
        nested.rmdir()
elif (DATA_DIR / "images").is_dir() and (DATA_DIR / "labels").is_dir():
    print(f"Архив не найден, использую уже распакованный датасет: {DATA_DIR}")
else:
    raise FileNotFoundError(
        "Не найден processed.zip и нет data/processed/{images,labels}. "
        "Загрузите архив в корень проекта или в data/."
    )

yaml_path = DATA_DIR / "data.yaml"
yaml_path.write_text(
    "\n".join(
        [
            f"path: {DATA_DIR.resolve().as_posix()}",
            "train: images/train",
            "val: images/val",
            "",
            "nc: 4",
            "names:",
            "  0: excavator",
            "  1: tower_crane",
            "  2: concrete_mixer",
            "  3: dump_truck",
            "",
        ]
    ),
    encoding="utf-8",
)

n_train = len(list((DATA_DIR / "images" / "train").glob("*")))
n_val = len(list((DATA_DIR / "images" / "val").glob("*")))
print(f"images: train={n_train}, val={n_val}")
print(f"data.yaml: {yaml_path}")
print(yaml_path.read_text(encoding="utf-8"))

## 4. Запуск обучения

`ml/train.py` с параметрами `epochs=100`, `batch=16`, `imgsz=640`.

In [ ]:
import os
import sys

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.train import train

best_pt = train(
    weights="yolov8n.pt",
    data=DATA_DIR / "data.yaml",
    epochs=100,
    batch=16,
    imgsz=640,
    patience=20,
    device=None,  # авто: cuda при наличии GPU
    project=PROJECT_ROOT / "ml" / "runs" / "detect",
    name="train_datasphere",
)

print("best_pt =", best_pt)

## 5. Финальные метрики

Метрики уже печатает `train()`. Ниже — повторная валидация `best.pt` для удобного просмотра в ноутбуке.

In [ ]:
from ultralytics import YOLO
from ml.train import print_metrics, detect_device

device = detect_device()
val_model = YOLO(str(best_pt))
val_results = val_model.val(
    data=str((DATA_DIR / "data.yaml").resolve()),
    imgsz=640,
    batch=16,
    device=device,
    split="val",
)
print_metrics(val_results)

## 6. Сохранение `best.pt` в удобное место

Копия кладётся в `outputs/best.pt` в корне проекта — оттуда удобно скачать через File Browser.

In [ ]:
from pathlib import Path
import shutil

out_dir = PROJECT_ROOT / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_best = out_dir / "best.pt"

src = Path(best_pt)
if not src.is_file():
    raise FileNotFoundError(f"best.pt не найден: {src}")

shutil.copy2(src, out_best)
print(f"saved: {out_best}")
print(f"size:  {out_best.stat().st_size / 1e6:.2f} MB")
print(f"source: {src.resolve()}")